In [2]:
# ==============================
# Import Required Libraries
# ==============================

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

ModuleNotFoundError: No module named 'matplotlib'

# Feature Extraction and Price Prediction for Mobile Phones

## Project Objective

The objective of this project is to identify the key features affecting mobile phone prices and develop a machine learning model that accurately predicts mobile phone prices based on their specifications.

In [ ]:
df = pd.read_csv(r"C:\Users\MEGHA SHARMA\Downloads\Processed_Flipdata - Processed_Flipdata (2).csv")

In [ ]:
df

Insight

Previewing the dataset helps verify that the data has been imported correctly and provides an initial understanding of the available features.

In [ ]:
# Shape of Dataset
print("Number of Rows :", df.shape[0])
print("Number of Columns :", df.shape[1])

insight

A dataset with over 500 observations is adequate for developing and evaluating machine learning regression models.

In [ ]:
# Display Column Names
df.columns

In [ ]:
# Dataset Information
df.info()

Insight

A clean dataset with complete records reduces preprocessing effort and improves model reliability.

In [ ]:
# Statistical Summary
df.describe().T

Observation

The numerical variables show different value ranges, indicating that feature scaling may be required for algorithms sensitive to feature magnitude.

Business Insight

Understanding the statistical distribution of numerical variables helps identify unusual values and supports feature engineering decisions.

In [ ]:
# Missing Values
df.isnull().sum()

In [ ]:
# Duplicate Records
print("Duplicate Records :", df.duplicated().sum())

Observation

The dataset does not contain duplicate records.

In [ ]:
# Unique Values in Categorical Columns
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"{col} : {df[col].nunique()} unique values")

# Data Cleaning

In [ ]:
# Remove Unnecessary Column

df.drop("Unnamed: 0", axis=1, inplace=True)

df.head()

Insight

Removing irrelevant columns helps reduce noise and improves the efficiency of the machine learning model.

# Convert Prize into Numeric

In [ ]:
df["Prize"].head(10)

In [ ]:
# Convert Prize to Numeric

df["Prize"] = (
    df["Prize"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [ ]:
df["Prize"].dtype

Insight

Machine learning regression models require the target variable to be numeric for accurate prediction.

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# Exploratory Data Analysis (EDA)

In [ ]:
# Numerical Columns
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns
print("Numerical Columns:")
print(numerical_columns)

print("-"*50)

# Categorical Columns
categorical_columns = df.select_dtypes(include="object").columns
print("Categorical Columns:")
print(categorical_columns)

Insight

Numerical features help analyze price trends, while categorical features help understand the impact of product specifications on pricing.

In [ ]:
# Distribution Plot

plt.figure(figsize=(15,10))

for i, col in enumerate(numerical_columns, 1):

    plt.subplot(2,3,i)

    sns.histplot(df[col], kde=True)

    plt.title(col)

plt.tight_layout()

plt.show()

Insight

Understanding the distribution of numerical variables helps identify dominant configurations and pricing patterns in the smartphone market.

# Boxplot (Outlier Visualization)

In [ ]:
plt.figure(figsize=(15,10))

for i, col in enumerate(numerical_columns,1):

    plt.subplot(2,3,i)

    sns.boxplot(y=df[col])

    plt.title(col)

plt.tight_layout()

plt.show()

# Countplot for AI Lens

In [ ]:
# plt.figure(figsize=(5,4))

sns.countplot(x=df["AI Lens"])

plt.title("AI Lens Distribution")

plt.show()

# Bivariate Analysis

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="Memory",
    y="Prize",
    data=df
)

plt.title("Memory vs Mobile Price")
plt.xlabel("Memory (GB)")
plt.ylabel("Price")

plt.show()

Insight

Memory is one of the key factors influencing smartphone pricing, as larger storage options are associated with premium devices.

# RAM vs Prize

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x="RAM",
    y="Prize",
    data=df
)

plt.title("RAM vs Mobile Price")
plt.xlabel("RAM (GB)")
plt.ylabel("Price")

plt.show()

# Battery vs Prize

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    x="Battery_",
    y="Prize",
    data=df
)

plt.title("Battery Capacity vs Price")

plt.show()

Insight

Although larger batteries add value, battery capacity alone does not determine the overall smartphone price.

# Mobile Height vs Prize

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    x="Mobile Height",
    y="Prize",
    data=df
)

plt.title("Mobile Height vs Price")

plt.show()

Observation

Mobile height has a weak relationship with smartphone price.

# Processor vs Prize

In [ ]:
plt.figure(figsize=(14,6))

top_processor = df["Processor_"].value_counts().head(10).index

sns.boxplot(
    x="Processor_",
    y="Prize",
    data=df[df["Processor_"].isin(top_processor)]
)

plt.xticks(rotation=90)

plt.title("Top 10 Processors vs Price")

plt.show()

Insight

Processor performance is a major contributor to smartphone pricing and plays a significant role in market segmentation.

# Correlation Heatmap

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

Observation

Memory and RAM show stronger positive correlations with price compared to other numerical features.

Business Insight

Highly correlated features are valuable predictors for developing an accurate price prediction model.

# Feature Engineering

In [ ]:
# Features and Target Variable

X = df.drop(["Prize", "Model"], axis=1)

y = df["Prize"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Insight

Removing high-cardinality identifier columns helps reduce unnecessary complexity and improves model generalization.

# One-Hot Encoding

In [ ]:
X = pd.get_dummies(
    X,
    columns=[
        "Colour",
        "Rear Camera",
        "Front Camera",
        "Processor_"
    ],
    drop_first=True
)

print("Shape after Encoding :", X.shape)

Observation

Categorical variables have been converted into numerical format using One-Hot Encoding.

Business Insight

Machine learning algorithms require numerical inputs; therefore, encoding categorical features is essential.

# Variance Threshold (Feature Selection)

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)

X_selected = selector.fit_transform(X)

selected_columns = X.columns[selector.get_support()]

X = pd.DataFrame(
    X_selected,
    columns=selected_columns
)

print("Final Shape :", X.shape)

Observation

Low-variance features were removed, reducing the feature space while retaining the most informative variables.

Business Insight

Eliminating low-variance features reduces model complexity and improves computational efficiency.

In [ ]:
X.head()

# Observation

The dataset is now fully prepared for machine learning model development.

Business Insight

The refined feature set contains only relevant variables, which helps improve prediction performance.

# Model Building

In [ ]:
# Split Original Data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Target :", y_train.shape)
print("Testing Target :", y_test.shape)

Observation

The dataset has been successfully divided into training and testing sets using an 80:20 ratio.

Business Insight

Splitting the data ensures that the model is evaluated on unseen data, providing a realistic measure of prediction performance

# Scaling (Only for Linear Regression)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

Observation

Feature scaling has been applied only for Linear Regression.

Business Insight

Scaling ensures that numerical features contribute equally to distance-based and linear algorithms.

# Linear Regression

In [ ]:
lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_pred)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression Performance")
print("-"*40)

print("MAE :", round(lr_mae,2))
print("RMSE :", round(lr_rmse,2))
print("R2 Score :", round(lr_r2,4))

# Insight

The baseline model helps measure the improvement achieved by advanced machine learning algorithms.

In [ ]:
dt = DecisionTreeRegressor(random_state=42)

dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

dt_mae = mean_absolute_error(y_test, dt_pred)

dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

dt_r2 = r2_score(y_test, dt_pred)

print("Decision Tree Performance")
print("-"*40)

print("MAE :", round(dt_mae,2))
print("RMSE :", round(dt_rmse,2))
print("R2 Score :", round(dt_r2,4))

Observation

Decision Tree captures complex non-linear relationships between smartphone specifications and price.

Business Insight

Tree-based models are effective in identifying interactions among multiple hardware features.

# Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Performance")
print("-"*40)

print("MAE :", round(rf_mae,2))
print("RMSE :", round(rf_rmse,2))
print("R2 Score :", round(rf_r2,4))

# Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(random_state=42)

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_pred)

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))

gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting Performance")
print("-"*40)

print("MAE :", round(gb_mae,2))
print("RMSE :", round(gb_rmse,2))
print("R2 Score :", round(gb_r2,4))

# Model Comparison

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],

    "MAE":[
        lr_mae,
        dt_mae,
        rf_mae,
        gb_mae
    ],

    "RMSE":[
        lr_rmse,
        dt_rmse,
        rf_rmse,
        gb_rmse
    ],

    "R2 Score":[
        lr_r2,
        dt_r2,
        rf_r2,
        gb_r2
    ]

})

comparison = comparison.sort_values(
    by="R2 Score",
    ascending=False
)

comparison

Observation

Among all the regression models, the Decision Tree Regressor achieved the highest R² Score (94.96%) and the lowest MAE and RMSE values. This indicates that it provides the most accurate predictions for mobile phone prices on the test dataset.

Business Insight

The Decision Tree model effectively captures the complex relationship between smartphone specifications and pricing, making it the most suitable model for supporting pricing decisions in this project.

# Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(10),
    x="Importance",
    y="Feature"
)

plt.title("Top 10 Important Features Affecting Mobile Price")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.show()

Observation

Memory is the most influential feature in predicting mobile phone prices, followed by Front Camera (12MP), Mobile Height, and Battery Capacity. Camera specifications and RAM also contribute to price prediction but with comparatively lower importance.

 Business Insight

Smartphone pricing is primarily influenced by storage capacity, camera quality, and battery specifications. Companies should prioritize these hardware features while designing premium smartphones and determining pricing strategies.

# Actual vs Prediction

In [ ]:
prediction = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": dt_pred
})

prediction.head(10)

Observation 

The Decision Tree model predicts most mobile phone prices very close to the actual values. Only a few observations show noticeable prediction errors, indicating that the model has learned the pricing patterns effectively.

Business Insight

The model demonstrates high prediction accuracy and can be used to estimate the prices of newly launched smartphones based on their specifications. This can support pricing strategy and market analysis.

# Scatter PLot 

In [3]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, dt_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")

plt.show()

NameError: name 'plt' is not defined

Observation

Most data points are located close to the diagonal reference line, indicating that the predicted prices closely match the actual prices.

Business Insight

A strong alignment between actual and predicted prices demonstrates that the model can be used as a reliable pricing support tool.

# Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    DecisionTreeRegressor(random_state=420),
    X,
    y,
    cv=5,
    scoring="r2"
)

print("Cross Validation Scores")
print(cv_scores)

print("\nAverage R2 Score :", cv_scores.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

# Hyperparameter grid
param_grid_dt = {
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'criterion': ['squared_error', 'friedman_mse']
}

dt = DecisionTreeRegressor(random_state=42)

grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_dt.fit(X, y)

print("Best Parameters (Decision Tree):", grid_dt.best_params_)
print("Best CV R2 Score:", grid_dt.best_score_)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestRegressor(random_state=42)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X, y)

print("Best Parameters (Random Forest):", grid_rf.best_params_)
print("Best CV R2 Score:", grid_rf.best_score_)


# Final Model

In [ ]:
# Best Random Forest Model

best_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    random_state=42
)

best_rf.fit(X_train, y_train)

In [ ]:
best_pred = best_rf.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, best_pred)

rmse = np.sqrt(mean_squared_error(y_test, best_pred))

r2 = r2_score(y_test, best_pred)

print("Final Model Performance")
print("-"*35)

print("MAE :", round(mae,2))
print("RMSE :", round(rmse,2))
print("R2 Score :", round(r2,4))

In [ ]:
import joblib

joblib.dump(best_rf, "mobile_price_prediction_model.pkl")

# Predict New Mobile Price

In [ ]:
import joblib

loaded_model = joblib.load("mobile_price_prediction_model.pkl")

In [ ]:
sample = X.iloc[[0]]

prediction = loaded_model.predict(sample)

print("Predicted Price:", prediction[0])

# Business Recommendations

1. Memory is the most influential feature affecting mobile phone prices. The company should focus on offering higher storage variants for premium devices.

2. Camera specifications, especially front camera quality, significantly impact smartphone pricing and customer preference.

3. Battery capacity plays an important role in determining product value. Devices with larger batteries can be positioned in the premium segment.

4. RAM contributes to overall device performance and should be considered while designing smartphones for different customer segments.

5. The Random Forest and Decision Tree models can assist the organization in estimating competitive prices for newly launched smartphones.

6. Machine learning-based pricing can reduce manual effort and improve pricing consistency across different product categories.

# Conclusion

This project successfully analyzed the key factors influencing mobile phone prices and developed multiple machine learning regression models for price prediction.

After comparing the performance of Linear Regression, Decision Tree, Random Forest, and Gradient Boosting, the Decision Tree model achieved the highest Test R² Score of 94.96%. However, Random Forest demonstrated better generalization performance during cross-validation, making it a more reliable model for real-world deployment.

Feature importance analysis revealed that Memory, Front Camera specifications, Mobile Height, Battery Capacity, and RAM are the most influential factors affecting smartphone prices.

The developed machine learning solution can help organizations improve pricing strategies, estimate prices for new products, and support business decision-making.

# Limitations

- The dataset contains only 541 mobile phones, which limits the diversity of training data.
- Market demand, brand popularity, discounts, and customer reviews were not included.
- Prices may change over time due to market trends.
- The model performance depends on the quality and size of the available dataset.

# Future Scope

- Increase the dataset size for better model performance.
- Include additional features such as Brand Rating, Display Type, Refresh Rate, 5G Support, and Customer Reviews.
- Deploy the model as a web application using Flask or Streamlit.
- Retrain the model periodically with new smartphone data to improve prediction accuracy.